In [ ]:
import re
import time
import shutil
import unicodedata
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text, types as sat

🧩 Bloque 1 — Importación de librerías

Carga de dependencias para manejo de datos (pandas/numpy), rutas y conexión SQL.

Base para lectura, limpieza y escritura.

In [ ]:
#from pathlib import Path
import sys
import importlib

# --- Resolver rutas ---
# notebook: .../traspaso-innominados/new_source/new_notebooks
# queremos:  .../traspaso-innominados/functions
PROJECT_ROOT = Path.cwd().resolve().parents[1]
FUNCTIONS_DIR = (PROJECT_ROOT / "functions").resolve()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
# Si tu archivo se llama functions.py dentro de FUNCTIONS_DIR, esto funciona.
# Si es paquete "functions/" con __init__.py, esto también funciona.
module_name = "functions"

if module_name in sys.modules:
    fun = importlib.reload(sys.modules[module_name])
else:
    fun = importlib.import_module(module_name)

print("Importado desde:", getattr(fun, "__file__", "<sin __file__>"))

⚙️ Bloque 2 — Configuración de conexión a SQL Server

Parámetros: server, database, schema, table.

Creación del engine (SQLAlchemy) para consultas e inserciones.

In [ ]:
# --- Parámetros iguales a los tuyos ---
server = "SGF1034"
database = "Habitat"
schema = "dbo"
table = "VOLVEK_ACUMULADO_Planes"

# --- Nuevo: construir engine + obtener connection_string pyodbc ---
# Usa tus funciones: build_sqlserver_engine (y opcionalmente diagnósticos)

driver = "ODBC Driver 17 for SQL Server"  # o "ODBC Driver 18 for SQL Server"

# 1) ODBC connection string (pyodbc) -> para query_to_df / df_to_db / exec_sql
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# Si tu entorno requiere TLS/cert (muy común con Driver 18), descomenta:
# connection_string += "Encrypt=yes;TrustServerCertificate=yes;"

# 2) SQLAlchemy engine -> usando tu helper (hace test SELECT 1 y crea el engine)
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    # encrypt=True, trust_server_certificate=True,  # si lo necesitas
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise",  # "warn" si prefieres que no explote y solo imprima diagnóstico
)

📂 Bloque 3 — Ruta de archivos y hoja objetivo

Carpeta origen y extensiones permitidas.

Hojas preferidas y fallback por índice.

Permite localizar automáticamente el archivo/hoja a procesar.

In [ ]:
RUTA_ARCHIVOS = (PROJECT_ROOT / "data" / "CCLA" / "VOLVEK PLANES").resolve()


print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUTA_ARCHIVOS:", RUTA_ARCHIVOS, "exists:", RUTA_ARCHIVOS.exists())

#RUTA_ARCHIVOS = Path(r"C:\Users\aepnlizama\OneDrive - Seguros Suramericana, S.A\Escritorio\Programacion\Automatizaciones\Pruebas\CCLA\VOLVEK PLANES")
EXTS = (".xlsx",)  # Extensiones de archivos a considerar

PREFERRED_SHEETS = ["Base Planes", "planes", "Planes"]
FALLBACK_SHEET_INDEX = 0

📑 Bloque 5 — Selección del archivo más reciente

Verifica existencia de la ruta.

Filtra por extensión y ordena por última modificación.

Abre con copia temporal si el archivo está bloqueado (OneDrive, etc.).

In [5]:
assert RUTA_ARCHIVOS.is_dir(), f"No existe carpeta: {RUTA_ARCHIVOS}"
archivos = [p for p in RUTA_ARCHIVOS.iterdir() if p.is_file() and p.suffix.lower() in EXTS]
archivos.sort(key=lambda p: p.stat().st_mtime, reverse=True)
assert archivos, "No se encontraron Excel válidos."

archivo_origen = archivos[0]
print(f"📄 Archivo más reciente: {archivo_origen.name}  |  {datetime.fromtimestamp(archivo_origen.stat().st_mtime):%Y-%m-%d %H:%M:%S}")

_tmp_copy_path = None

def _norm(s): 
    return " ".join(str(s).split()).lower()

pref_norm = {_norm(n) for n in PREFERRED_SHEETS}

try:
    try:
        with pd.ExcelFile(archivo_origen, engine="openpyxl") as xls:
            print("Hojas:", xls.sheet_names)

            target = None
            for s in xls.sheet_names:
                if _norm(s) in pref_norm:
                    target = s
                    break
            if target is None:
                target = xls.sheet_names[FALLBACK_SHEET_INDEX]
            print(f"✅ Hoja objetivo: {target}")

    except PermissionError:
        _tmp_copy_path = archivo_origen.parent / f"__tmp_copy_{int(time.time()*1000)}{archivo_origen.suffix}"
        shutil.copy2(archivo_origen, _tmp_copy_path)
        print(f"📂 No se pudo abrir el archivo original, usando copia temporal: {_tmp_copy_path.name}")

        with pd.ExcelFile(_tmp_copy_path, engine="openpyxl") as xls:
            print("Hojas:", xls.sheet_names)

            target = None
            for s in xls.sheet_names:
                if _norm(s) in pref_norm:
                    target = s
                    break
            if target is None:
                target = xls.sheet_names[FALLBACK_SHEET_INDEX]
            print(f"✅ Hoja objetivo (copia): {target}")

except Exception as e:
    raise SystemExit(f"❌ Error al abrir el Excel: {e}")

📄 Archivo más reciente: Base cesantia SURA Planes NOVIEMBRE 2025.xlsx  |  2025-12-29 16:34:55
Hojas: ['Hoja1', 'Planes']
✅ Hoja objetivo: Planes


🧾 Bloque 6 — Lectura del Excel y limpieza básica

Lectura segura (dtype=str) para evitar pérdidas de formato.
Limpieza de valores vacíos comunes.
Normalización de encabezados con fun.normalize_name.

In [ ]:
def read_excel_safe(io, sheet_name):
    try:
        return pd.read_excel(io, sheet_name=sheet_name, dtype=str, engine="openpyxl")
    except Exception as e:
        raise SystemExit(f"No se pudo leer la hoja '{sheet_name}': {e}")

io = _tmp_copy_path if _tmp_copy_path else archivo_origen
df_raw = read_excel_safe(io, target)

for c in df_raw.columns:
    if df_raw[c].dtype == object:
        df_raw[c] = df_raw[c].astype(str).str.strip().replace({"None": "", "none": "", "NaN": "", "nan": ""})
df_raw.columns = [fun.normalize_name(c) for c in df_raw.columns]

print("Encabezados normalizados:", list(df_raw.columns))

df_raw.head()

Encabezados normalizados: ['noperacion', 'afinom', 'afirut', 'afirutdv', 'crecuotot', 'cresolmon', 'fecinicob', 'fectercob', 'crecuomon', 'prima_cliente', 'tasa', 'plan', 'prima_exenta', 'comision_caja_25', 'producto', 'folio_original', 'fecha_otorgamiento_folio_original']


,noperacion,afinom,afirut,afirutdv,crecuotot,cresolmon,fecinicob,fectercob,crecuomon,prima_cliente,tasa,plan,prima_exenta,comision_caja_25,producto,folio_original,fecha_otorgamiento_folio_original
0,136000203058,DAFNE AGUILERA CAMPOS,16692442,1,37,2132125,20240822,20270930,50581,2324,0.109,Plan 4 Cuotas,1952.9411764705883,488.2352941176471,REPROGRAMACION,136000189359,20180524
1,134000381606,JUAN ORTIZ QUINTANA,17621935,1,60,9580551,20250701,20300731,141283,18490,0.193,Plan 8 Cuotas,15537.81512605042,3884.453781512605,REPROGRAMACION,134000365938,20180326
2,041000403541,PATRICIO GONZALEZ CASTILLO,9349593,4,57,6773967,20210531,20260228,125063,13097,0.193,Plan 8 Cuotas,11005.882352941177,2751.470588235294,REPROGRAMACION,241000034673,20170821
3,001027701039,GERMAN NARANJO SANDOVAL,10157539,K,60,2027725,20240619,20290630,48100,3914,0.193,Plan 8 Cuotas,3289.075630252101,822.2689075630252,REPROGRAMACION,008000518808,20180419
4,085000200303,PATRIC MOLINA VIILLAGRAN,19514790,6,88,3887153,20180731,20251130,73983,9813,0.252,Plan 12 Cuotas,8246.218487394959,2061.5546218487398,REPROGRAMACION,001027663597,20170830


🗂️ Bloque 7 — Mapeo de columnas al modelo estándar

Uso de pick() para tolerar sinónimos de encabezados.
Construcción de df_can con nombres alineados al modelo SQL.

In [7]:
def pick(df, *names):
    for n in names:
        if n in df.columns:
            return df[n]
    return pd.Series([None]*len(df), index=df.index)

# Origen → Destino
foliocredito        = pick(df_raw, "n_operacion", "no_operacion", "noperacion", "operacion", "folio")
NombreAfiliado      = pick(df_raw, "afinom", "nombre_afiliado", "nombre")
rutafiliado         = pick(df_raw, "afirut", "rut_afiliado", "rut")
dvafiliado          = pick(df_raw, "afirutdv", "dv_afiliado", "dv")
Plazo               = pick(df_raw, "crecuotot", "plazo")
MontoBruto          = pick(df_raw, "cresolmon", "monto_bruto", "monto")
fechaPrimerVto      = pick(df_raw, "fecinicob", "fecha_primer_vto", "fec_ini_cob")
FechaUltimoVto      = pick(df_raw, "fectercob", "fecha_ultimo_vto", "fec_ter_cob")
ValorCuota          = pick(df_raw, "crecuomon", "valor_cuota")
PrimaBruta_src      = pick(df_raw, "prima_cliente", "prima_bruta", "prima_cliente_")
Tasa                = pick(df_raw, "tasa_interes", "tasa")  
Planes              = pick(df_raw, "plan")
PrimaNeta_src       = pick(df_raw, "prima_exenta", "prima_neta")
Com25_src           = pick(df_raw, "comision_caja_25", "comision_caja_25_", "comision_25")
Com26_src           = pick(df_raw, "comision_caja_variable_39", "comision_variable_39", "comision_39")
Producto            = pick(df_raw, "producto")
FolioOrigen         = pick(df_raw, "folio_original", "folio_origen")
FechaOrigen         = pick(df_raw, "fecha_otorgamiento_folio_original", "fecha_origen")


df_can = pd.DataFrame({
    "foliocredito": foliocredito,
    "NombreAfiliado": NombreAfiliado,
    "rutafiliado": rutafiliado,
    "dvafiliado": dvafiliado,
    "Plazo": Plazo,
    "MontoBruto": MontoBruto,
    "fechaPrimerVto": fechaPrimerVto,
    "FechaUltimoVto": FechaUltimoVto,
    "ValorCuota": ValorCuota,
    "PrimaBruta": PrimaBruta_src,
    "Tasa": Tasa,
    "Planes": Planes,
    "PrimaNeta": PrimaNeta_src,
    "Comision25": Com25_src,
    "ComisionVariable": Com26_src,
    "Producto": Producto,
    "FolioOrigen": FolioOrigen,
    "FechaOrigen": FechaOrigen,
    "Nombre_de_archivo": archivo_origen.name,
})

df_can.head()

,foliocredito,NombreAfiliado,rutafiliado,dvafiliado,Plazo,MontoBruto,fechaPrimerVto,FechaUltimoVto,ValorCuota,PrimaBruta,Tasa,Planes,PrimaNeta,Comision25,ComisionVariable,Producto,FolioOrigen,FechaOrigen,Nombre_de_archivo
0,136000203058,DAFNE AGUILERA CAMPOS,16692442,1,37,2132125,20240822,20270930,50581,2324,0.109,Plan 4 Cuotas,1952.9411764705883,488.2352941176471,None,REPROGRAMACION,136000189359,20180524,Base cesantia SURA Planes NOVIEMBRE 2025.xlsx
1,134000381606,JUAN ORTIZ QUINTANA,17621935,1,60,9580551,20250701,20300731,141283,18490,0.193,Plan 8 Cuotas,15537.81512605042,3884.453781512605,None,REPROGRAMACION,134000365938,20180326,Base cesantia SURA Planes NOVIEMBRE 2025.xlsx
2,041000403541,PATRICIO GONZALEZ CASTILLO,9349593,4,57,6773967,20210531,20260228,125063,13097,0.193,Plan 8 Cuotas,11005.882352941177,2751.470588235294,None,REPROGRAMACION,241000034673,20170821,Base cesantia SURA Planes NOVIEMBRE 2025.xlsx
3,001027701039,GERMAN NARANJO SANDOVAL,10157539,K,60,2027725,20240619,20290630,48100,3914,0.193,Plan 8 Cuotas,3289.075630252101,822.2689075630252,None,REPROGRAMACION,008000518808,20180419,Base cesantia SURA Planes NOVIEMBRE 2025.xlsx
4,085000200303,PATRIC MOLINA VIILLAGRAN,19514790,6,88,3887153,20180731,20251130,73983,9813,0.252,Plan 12 Cuotas,8246.218487394959,2061.5546218487398,None,REPROGRAMACION,001027663597,20170830,Base cesantia SURA Planes NOVIEMBRE 2025.xlsx


🏷️ Bloque 8 — Nombre canónico del archivo (YYYYMM)

Extracción del período desde el nombre (o fallback con timestamp).
Asignación a Nombre_de_archivo para trazabilidad.

In [ ]:
MESES = {
    "ENERO": "01", "ENE": "01",
    "FEBRERO": "02", "FEB": "02",
    "MARZO": "03", "MAR": "03",
    "ABRIL": "04", "ABR": "04",
    "MAYO": "05", "MAY": "05",
    "JUNIO": "06", "JUN": "06",
    "JULIO": "07", "JUL": "07",
    "AGOSTO": "08", "AGO": "08",
    "SEPTIEMBRE": "09", "SETIEMBRE": "09", "SEP": "09", "SET": "09",
    "OCTUBRE": "10", "OCT": "10",
    "NOVIEMBRE": "11", "NOV": "11",
    "DICIEMBRE": "12", "DIC": "12",
}

def _extract_yyyymm_from_name(nombre: str) -> str:
    """
    Intenta extraer YYYYMM desde el nombre del archivo (sin importar la extensión).
    Cubre:
      - YYYYMM
      - YYYY[-_/ .]?MM
      - MM[-_/ .]?YYYY
      - MES(ES) + YYYY (con tildes/abreviaturas) en cualquier orden
    Lanza ValueError si no encuentra período.
    """
    stem = Path(nombre).stem
    stem_norm = fun._strip_accents(stem).upper()

    m = re.search(r"(20\d{2})(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"

    m = re.search(r"(20\d{2})[-_/.\s]?(0[1-9]|1[0-2])", stem_norm)
    if m:
        return f"{m.group(1)}{m.group(2)}"

    m = re.search(r"(0[1-9]|1[0-2])[-_/.\s]?(20\d{2})", stem_norm)
    if m:
        return f"{m.group(2)}{m.group(1)}"

    m_year = re.search(r"(20\d{2})", stem_norm)
    if m_year:
        year = m_year.group(1)
        # Aseguramos coincidencias completas de palabra para evitar falsos positivos (e.g. "MAR" en "MARCHA")
        for mes_txt, mm in MESES.items():
            # \b no siempre funciona con underscores, por eso agregamos delimitadores comunes
            if re.search(rf"(?<![A-Z0-9]){mes_txt}(?![A-Z0-9])", stem_norm):
                return f"{year}{mm}"

    raise ValueError(f"No pude extraer YYYYMM desde el nombre: {nombre}")

def canonicalizar_planes(nombre: str) -> str:
    """
    Devuelve exactamente: 'SURA Planes YYYYMM.xlsx'
    (fuerza la extensión .xlsx sin importar la original)
    """
    yyyymm = _extract_yyyymm_from_name(nombre)
    return f"SURA Planes {yyyymm}.xlsx"


nombre_canonico = canonicalizar_planes(archivo_origen.name)
print("Nombre original :", archivo_origen.name)
print("Nombre canónico :", nombre_canonico)

df_can["Nombre_de_archivo"] = nombre_canonico


Nombre original : Base cesantia SURA Planes NOVIEMBRE 2025.xlsx
Nombre canónico : SURA Planes 202511.xlsx


🧮 Bloque 9 — Eliminación de fila de totales

Detección de fila final de sumas (coincidencia con sumatorias y alta nulidad).

Eliminación segura y registro en consola.

In [9]:
def _is_nullish(v):
    if pd.isna(v):
        return True
    if isinstance(v, str):
        s = v.strip()
        return (s == "") or (s.lower() == "none")
    return False


def _nullish_ratio_last_row(df, exclude=()):
    cols = [c for c in df.columns if c not in exclude]
    if not cols:
        return 1.0
    last = df.iloc[-1]
    n_null = sum(_is_nullish(last[c]) for c in cols)
    return n_null / len(cols)


def _round_safe(x, nd):
    try:
        return round(float(x), nd)
    except Exception:
        return np.nan


def drop_last_total_strict(
    df: pd.DataFrame,
    money_cols=("PrimaBruta","PrimaNeta","Comision25"),
    round_decimals={"PrimaBruta": 0, "PrimaNeta": 2, "Comision25": 2},
    null_check_exclude=("Nombre_de_archivo",),
    null_ratio_threshold=0.80,
    atol=0.5,
    rtol=5e-4,
):
    """
    Elimina la última fila si:
      A) Para cada col monetaria: round(last, nd) == round(sum(prev), nd)  O  isclose con (rtol, atol)
      B) En las demás columnas (excepto las excluidas), tiene >= null_ratio_threshold de nulos.
    Muestra la fila eliminada en consola.
    """
    if df is None or df.empty or len(df) < 2:
        print("DF vacío o con <2 filas: nada que eliminar.")
        return df

    last_idx = df.index[-1]
    prev = df.iloc[:-1]
    last = df.iloc[-1]

    # --- A) Chequeo de sumas ---
    sum_flags = {}
    details = {}
    for c in money_cols:
        if c not in df.columns:
            sum_flags[c] = False
            details[c] = "columna no existe"
            continue

        prev_sum = pd.to_numeric(prev[c], errors="coerce").sum(skipna=True)
        last_val = pd.to_numeric(pd.Series([last[c]]), errors="coerce").iloc[0]

        nd = round_decimals.get(c, 2)
        prev_sum_r = _round_safe(prev_sum, nd)
        last_val_r = _round_safe(last_val, nd)

        ok_round = (not np.isnan(prev_sum_r)) and (not np.isnan(last_val_r)) and (last_val_r == prev_sum_r)
        ok_close = (
            np.isfinite(prev_sum) and np.isfinite(last_val) and
            np.isclose(float(last_val), float(prev_sum), rtol=rtol, atol=atol)
        )
        sum_flags[c] = bool(ok_round or ok_close)
        details[c] = f"last={last_val} (r{nd}={last_val_r}) vs sum(prev)={prev_sum} (r{nd}={prev_sum_r})"

    all_sums_ok = all(sum_flags.values())

    # --- B) Chequeo de nulos ---
    null_ratio = _nullish_ratio_last_row(df[[c for c in df.columns if c not in money_cols]], exclude=null_check_exclude)
    nulls_ok = (null_ratio >= null_ratio_threshold)

    # --- C) Decisión ---
    if all_sums_ok and nulls_ok:
        print("🧹 Última fila TOTAL detectada (sumas OK y muchos nulos). Eliminando…\n")
        removed = df.loc[[last_idx]].copy()

        print("🧾 Fila eliminada:")
        display(removed)

        out = df.iloc[:-1].reset_index(drop=True)
        return out

    print("❎ No se eliminó la última fila.")
    print(" - Sumas por columna:")
    for c in money_cols:
        if c in sum_flags:
            print(f"   {c}: match={sum_flags[c]} | {details.get(c)}")
        else:
            print(f"   {c}: columna no existe")
    print(f" - Null ratio última fila (no monetarias): {null_ratio:.2%} (umbral {null_ratio_threshold:.0%})")
    return df

df_can = drop_last_total_strict(df_can)

🧹 Última fila TOTAL detectada (sumas OK y muchos nulos). Eliminando…

🧾 Fila eliminada:


,foliocredito,NombreAfiliado,rutafiliado,dvafiliado,Plazo,MontoBruto,fechaPrimerVto,FechaUltimoVto,ValorCuota,PrimaBruta,Tasa,Planes,PrimaNeta,Comision25,ComisionVariable,Producto,FolioOrigen,FechaOrigen,Nombre_de_archivo
258,,,,,,,,,,1254423,,,1054136.9747899151,263534.2436974788,None,,,,SURA Planes 202511.xlsx


🔢 Bloque 10 — Tipificación y validación final

Conversión de tipos: Int64, float64, string.

Limpieza de dvafiliado, longitudes de texto y nulos críticos.

Orden de columnas para df_sql.

In [10]:
# --- 1) Tipos esperados por columna (según tu tabla) ---
INT_COLS    = ["rutafiliado", "Plazo", "fechaPrimerVto", "FechaUltimoVto", "FechaOrigen"]
BIGINT_COLS = ["foliocredito", "FolioOrigen"]
FLOAT_COLS  = ["MontoBruto", "ValorCuota", "PrimaBruta", "PrimaNeta", "Comision25", "Tasa","ComisionVariable"]
STR_COLS    = ["NombreAfiliado", "Producto", "Planes", "Nombre_de_archivo"]
DV_COL      = "dvafiliado"


def _to_num_series(s: pd.Series) -> pd.Series:
    if not pd.api.types.is_object_dtype(s):
        return pd.to_numeric(s, errors="coerce")
    s2 = s.astype(str).str.strip().replace({"": np.nan, "None": np.nan, "none": np.nan})
    return pd.to_numeric(s2, errors="coerce")

# BIGINT -> pandas Int64 (acepta NaN)
for c in BIGINT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("Int64")

# INT -> pandas Int64
for c in INT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("Int64")

# FLOAT -> float64
for c in FLOAT_COLS:
    if c in df_can.columns:
        df_can[c] = _to_num_series(df_can[c]).astype("float64")

# DV: char(1) en mayúscula
if DV_COL in df_can.columns:
    df_can[DV_COL] = (
        df_can[DV_COL]
        .astype("string")
        .str.strip()
        .str.upper()
        .map(lambda x: x[:1] if pd.notna(x) and len(x) > 0 else pd.NA)
    )

if "NombreAfiliado" in df_can.columns:
    df_can["NombreAfiliado"] = df_can["NombreAfiliado"].astype("string").str.strip().str.slice(0, 510)

if "Producto" in df_can.columns:
    df_can["Producto"] = df_can["Producto"].astype("string").str.strip().str.slice(0, 510)

if "Planes" in df_can.columns:
    df_can["Planes"] = df_can["Planes"].astype("string").str.strip().str.slice(0, 510)

if "Nombre_de_archivo" in df_can.columns:
    df_can["Nombre_de_archivo"] = df_can["Nombre_de_archivo"].astype("string").str.strip().str.slice(0, 50)

print("✅ dtypes DESPUÉS:\n", df_can.dtypes)

criticas = ["foliocredito","rutafiliado","dvafiliado","FolioOrigen","Nombre_de_archivo"]
presentes = [c for c in criticas if c in df_can.columns]
if presentes:
    print("\n🔎 Nulos en columnas críticas:")
    for c in presentes:
        print(f" - {c}: {int(df_can[c].isna().sum())} nulos")

cols_sql = [
    "foliocredito","NombreAfiliado","rutafiliado","dvafiliado","Plazo","MontoBruto",
    "fechaPrimerVto","FechaUltimoVto","ValorCuota","PrimaBruta", "Tasa", "Planes", "PrimaNeta",
    "Comision25","ComisionVariable","Producto","FolioOrigen","FechaOrigen","Nombre_de_archivo"
]

df_sql = df_can[[c for c in cols_sql if c in df_can.columns]].copy()

print("\n📦 df_sql listo con columnas:", list(df_sql.columns))

✅ dtypes DESPUÉS:
 foliocredito                  Int64
NombreAfiliado       string[python]
rutafiliado                   Int64
dvafiliado                   object
Plazo                         Int64
MontoBruto                  float64
fechaPrimerVto                Int64
FechaUltimoVto                Int64
ValorCuota                  float64
PrimaBruta                  float64
Tasa                        float64
Planes               string[python]
PrimaNeta                   float64
Comision25                  float64
ComisionVariable            float64
Producto             string[python]
FolioOrigen                   Int64
FechaOrigen                   Int64
Nombre_de_archivo    string[python]
dtype: object

🔎 Nulos en columnas críticas:
 - foliocredito: 0 nulos
 - rutafiliado: 0 nulos
 - dvafiliado: 0 nulos
 - FolioOrigen: 0 nulos
 - Nombre_de_archivo: 0 nulos

📦 df_sql listo con columnas: ['foliocredito', 'NombreAfiliado', 'rutafiliado', 'dvafiliado', 'Plazo', 'MontoBruto', 'fechaPri

C:\Users\aepnlizama\AppData\Local\Temp\ipykernel_10812\2329058015.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  s2 = s.astype(str).str.strip().replace({"": np.nan, "None": np.nan, "none": np.nan})


🧾 Bloque 11 – Validación de NOMBRE_ARCHIVO y conteo de filas por archivo

Este bloque valida:
- Que NOMBRE_ARCHIVO exista.
- Que tenga valores válidos (no vacíos).
- Cuenta cuántas filas trae cada archivo en el DataFrame.

Genera un diccionario {archivo → cantidad de filas}, usado luego para control de versiones.

In [11]:
assert "Nombre_de_archivo" in df_sql.columns, "Falta la columna 'Nombre_de_archivo' en df_sql."

df_sql["Nombre_de_archivo"] = (
    df_sql["Nombre_de_archivo"]
    .astype("string")
    .str.strip()
)

vals = [
    v for v in df_sql["Nombre_de_archivo"].dropna().unique()
    if str(v).strip() != ""
]

if not vals:
    raise SystemExit("🚨 No se encontraron valores válidos en 'Nombre_de_archivo'.")

counts = (
    df_sql["Nombre_de_archivo"]
    .value_counts(dropna=True)
    .to_dict()
)

print(f"📄 Se detectaron {len(vals)} Nombre_de_archivo distintos en el df_sql:")
for nombre, cnt in counts.items():
    print(f"   - {nombre}: {cnt} filas")

📄 Se detectaron 1 Nombre_de_archivo distintos en el df_sql:
   - SURA Planes 202511.xlsx: 258 filas


🔄 Bloque 12 – Carga con reemplazo selectivo por archivo (ETL transaccional)

Este bloque implementa el core del proceso ETL:
- Se ejecuta una sola transacción.
- Para cada NOMBRE_ARCHIVO:
- Consulta si existe data previa en SQL.
- Si existe → elimina solo ese archivo (reemplazo controlado).
- Inserta solo las filas nuevas.
- Imprime un resumen:
    - Filas reemplazadas
    - Filas insertadas
    - Archivos nuevos vs antiguos

Es un flujo robusto tipo "upsert por archivo".

In [12]:
resumen = []

with engine.begin() as conn:
    qry_count = text(f"""
        SELECT COUNT(*) AS cnt
        FROM {schema}.{table}
        WHERE Nombre_de_archivo = :nombre
    """)

    qry_del = text(f"""
        DELETE FROM {schema}.{table}
        WHERE Nombre_de_archivo = :nombre
    """)

    for nombre_archivo in vals:
        df_sub = df_sql[df_sql["Nombre_de_archivo"] == nombre_archivo]

        if df_sub.empty:
            print(f"⚠️ No se encontraron filas en df_sql para Nombre_de_archivo = '{nombre_archivo}'. Se omite.")
            continue

        existing_count = conn.execute(qry_count, {"nombre": nombre_archivo}).scalar() or 0

        if existing_count > 0:
            print(f"♻️ Se encontró data previa para Nombre_de_archivo = '{nombre_archivo}' "
                  f"({existing_count} filas en {schema}.{table}).")
            print("🧹 Eliminando filas anteriores para reemplazarlas por la nueva versión...")

            deleted = conn.execute(qry_del, {"nombre": nombre_archivo}).rowcount
            print(f"✅ Filas eliminadas en destino para '{nombre_archivo}': {deleted}")
            accion = "reemplazo"
        else:
            print(f"🆕 No hay data previa para Nombre_de_archivo = '{nombre_archivo}'. "
                  "Se insertará como archivo nuevo.")
            accion = "inserción nueva"

        df_sub.to_sql(
            name=table,
            con=conn,      
            schema=schema,
            if_exists='append',
            index=False
        )

        filas_sub = len(df_sub)
        print(f"📥 Insertadas {filas_sub} filas para Nombre_de_archivo = '{nombre_archivo}'.")
        resumen.append((nombre_archivo, filas_sub, existing_count, accion))


print("\n📊 Resumen de carga por Nombre_de_archivo:")
for nombre_archivo, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (reemplazando {existing_count} previas).")
    else:
        print(f"   - {nombre_archivo}: {filas_sub} filas cargadas (archivo nuevo).")

🆕 No hay data previa para Nombre_de_archivo = 'SURA Planes 202511.xlsx'. Se insertará como archivo nuevo.
📥 Insertadas 258 filas para Nombre_de_archivo = 'SURA Planes 202511.xlsx'.

📊 Resumen de carga por Nombre_de_archivo:
   - SURA Planes 202511.xlsx: 258 filas cargadas (archivo nuevo).


🗑️ Bloque 13 – Eliminación del archivo procesado

Finalmente:
- Una vez cargado en SQL, el archivo fuente se elimina automáticamente.
- Maneja errores comunes:
    - Archivo en uso por Excel/OneDrive
    - Permisos insuficientes
    - Casos en que el archivo no existe
    
Cierra el ciclo ETL manteniendo la carpeta siempre limpia.

In [13]:
try:
    if archivo_origen.exists() and archivo_origen.is_file():
        archivo_origen.unlink()
        print(f"\n🗑️ Archivo eliminado correctamente: {archivo_origen}")
    else:
        print(f"\n⚠️ No se pudo borrar el archivo porque no es un archivo válido: {archivo_origen}")
except PermissionError:
    print(f"\n⚠️ No se pudo borrar '{archivo_origen}': está en uso por OneDrive o Excel.")
except Exception as e:
    print(f"\n⚠️ Error inesperado al borrar archivo '{archivo_origen}': {e}")



🗑️ Archivo eliminado correctamente: C:\Users\aepnlizama\OneDrive - Seguros Suramericana, S.A\Escritorio\Programacion\Automatizaciones\Pruebas\CCLA\VOLVEK PLANES\Base cesantia SURA Planes OCTUBRE 2025.xlsx


# SQL

## VALIDACIONES / INSPECCIÓN

In [ ]:
query_1= """
-- [VOLVEK_PLANES] Archivos cargados (desc)
select distinct Nombre_de_archivo
from VOLVEK_ACUMULADO_Planes
order by Nombre_de_archivo desc;
"""

fun.exec_sql(query_1, connection_string)

## CONSTRUCCIÓN DE BASES FINALES POR CORREDOR / FUENTE

In [ ]:
query_2= """
drop table PLANES_FINAL_ACUMULADO;
"""

fun.exec_sql(query_2, connection_string)

In [ ]:
query_3= """
SELECT *,
       (CASE
            WHEN Planes IN ('Plan 03 Cuotas', 'Plan 3 Cuotas') THEN 4780715
            WHEN Planes IN ('Plan 04 Cuotas', 'Plan 4 Cuotas') THEN 4780716
            WHEN Planes IN ('Plan 06 Cuotas', 'Plan 6 Cuotas') THEN 4780717
            WHEN Planes IN ('Plan 08 Cuotas', 'Plan 8 Cuotas') THEN 4780718
            WHEN Planes = 'Plan 12 Cuotas' THEN 4780719
        END) as POLIZA,
       SUBSTRING(Nombre_de_archivo,
                 LEN(Nombre_de_archivo) - CHARINDEX('.', REVERSE(Nombre_de_archivo)) - 5,
                 6) AS MES_Recaudacion,
       (CASE
            WHEN Planes IN ('Plan 03 Cuotas', 'Plan 3 Cuotas') THEN 4331
            WHEN Planes IN ('Plan 04 Cuotas', 'Plan 4 Cuotas') THEN 4422
            WHEN Planes IN ('Plan 06 Cuotas', 'Plan 6 Cuotas') THEN 4332
            WHEN Planes IN ('Plan 08 Cuotas', 'Plan 8 Cuotas') THEN 4333
            WHEN Planes = 'Plan 12 Cuotas' THEN 4334
        END) as PLAN_TECNICO,
       (CASE
            WHEN Planes IN ('Plan 03 Cuotas', 'Plan 3 Cuotas') THEN 3
            WHEN Planes IN ('Plan 04 Cuotas', 'Plan 4 Cuotas') THEN 4
            WHEN Planes IN ('Plan 06 Cuotas', 'Plan 6 Cuotas') THEN 6
            WHEN Planes IN ('Plan 08 Cuotas', 'Plan 8 Cuotas') THEN 8
            WHEN Planes = 'Plan 12 Cuotas' THEN 12
        END) as PLAZO_CUOTAS,
       'Credito Consumo' as Negocio
INTO PLANES_FINAL_ACUMULADO
FROM VOLVEK_ACUMULADO_Planes;
"""

fun.exec_sql(query_3, connection_string)